In [ ]:
import pandas as pd

df = pd.read_csv("history.csv", low_memory=False)
date = df.loc[7920]["Date"]
def rollingForm(df: pd.DataFrame, date: pd.DatetimeIndex, season: int, HomeTeam: str, AwayTeam: str):
    # should output +- int 
    # pass in the date, HomeTeam, and AwayTeam
    # filter by the date, collect prev 5 home/away result

    # provides a df of the home team home games
    
    homeGames = df[(df["Date"] < date) & (df["HomeTeam"] == HomeTeam)].tail()
    awayGames = df[(df["Date"] < date) & (df["AwayTeam"] == AwayTeam)].tail()


    # include check for required min number of games
    return awayGames.head(1)

rollingForm(df,date, 2026 , "West Ham", "Leeds")

For promoted teams, we first have to convert their form from the championship to their predicted form in the prem
- this can be done thru linear regression -> feed the model the championship last 5 games form-> ask it to predict the form in the epl from that
How it can be done:
- first, finding the teams -> this is done when we try to find the team normally, check the season we are currently on and if cannot find the team in the previous season/ at all -> we look in the champioship file
- we take the last 5 results-> predict the home/away form it would be in the 5-10 games (evaluate which gives the best results) 
- as the matches occur -> update the rolling form (need a way to know if a team is promoted and if it still needs prev szn data and how much of it)
    - prior -> as new matches come add the new matches and find the avg rolling form 

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import log_loss, accuracy_score

prem = pd.read_csv("history.csv",low_memory=False)
cs = pd.read_csv("csHistory.csv",low_memory=False)

hf, premHF, af, premAF = [], [] ,[] ,[]
for i in range(2006, 2026):
    curSeason = prem[prem["season"] == i]
    promotedGames = curSeason[curSeason["HP"] == True]
    promotedTeams = promotedGames["HomeTeam"].unique()
    for team in promotedTeams:
        csLastSeason = cs[cs["season"] == (i - 1)]

        teamLast5HomeGames = csLastSeason[csLastSeason["HomeTeam"] == team].tail()
        
        homeForm = 0
        for game in teamLast5HomeGames.itertuples():
            # print(game)
            goals = game.FTHG
            homeForm += goals
        hf.append(homeForm)
            
        premFirstHomeGames = curSeason[curSeason["HomeTeam"] == team].head()
        premHomeForm = 0
        for game in premFirstHomeGames.itertuples():
            goals = game.FTHG
            premHomeForm += goals
        premHF.append(premHomeForm)

        teamLast5AwayGames = csLastSeason[csLastSeason["AwayTeam"] == team].tail()
        awayForm = 0
        for game in teamLast5AwayGames.itertuples():
            goals = game.FTAG
            awayForm += goals
        af.append(awayForm)

        premFirstAwayGames = curSeason[curSeason["AwayTeam"] == team].head()
        premAwayForm = 0
        for game in premFirstAwayGames.itertuples():
            goals = game.FTAG
            premAwayForm += goals
        premAF.append(premAwayForm)
        # put forms into a dict and convert it to a df
feats = {"HF": hf, "PremHF": premHF, "AF": af, "PremAF": premAF}
feats = pd.DataFrame(data=feats)


train = feats.head(45)

test = feats.tail(15)



trainXHome = train[["HF"]]
trainYHome = train[["PremHF"]]

testXHome = test[["HF"]]
testYHome = test[["PremHF"]]

model = LinearRegression()
model.fit(trainXHome,trainYHome)

model.score(testXHome,testYHome)

from sklearn.metrics import mean_absolute_error, mean_squared_error

pred = model.predict(testXHome)

mae = mean_absolute_error(testYHome, pred)
rmse = mean_squared_error(testYHome, pred) ** 0.5

baseline_pred = [trainYHome["PremHF"].mean()] * len(testYHome)

baseline_mae = mean_absolute_error(testYHome, baseline_pred)
baseline_rmse = mean_squared_error(testYHome, baseline_pred) ** 0.5

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

print("MAE:", mae)
print("RMSE:", rmse)

#     # find promoted teams in its respective season and update it
#     for promotedTeam in promotedTeams:
#         df.loc[(df["HomeTeam"] == promotedTeam) & (df["season"] == i), "HP"] = True
#         df.loc[(df["AwayTeam"] == promotedTeam) & (df["season"] == i), "AP"] = True
# prevTeams = curTeams

